In [2]:
import sys
sys.argv = [
    "program",
    "--input_path", "/workspace/Various-Model/data/sev.jsonl",
]


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
# MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
).eval()


def generate(emotion, scenario, event):
    messages = [
        {
            "role": "system",
            "content": (
                f"You are a human experiencing {emotion}. "
                f"Write 1–2 natural sentences expressing this emotion clearly."
            )
        },
        {
            "role": "user",
            "content": f"Scenario: {scenario}\nEvent: {event}"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only generated tokens
    gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    return gen_text.strip()


# =======================
# TEST SAMPLES
# =======================

scenario = "I completed the project presentation and submitted it to the team."
event = "The team praised my work and decided to implement my ideas."

print("MODEL:", MODEL_ID)

print("\nANGER:")
print(generate("anger", scenario, event))

print("\nSADNESS:")
print(generate("sadness", scenario, event))


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


MODEL: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B

ANGER:


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Okay, so the user is describing a situation where they've completed a project presentation and submission. The team praised their work and decided to implement their ideas. But the user is expressing anger in response. Hmm, why would they feel that way?

Maybe the user felt undervalued despite the positive feedback. Perhaps they expected the team to ignore their input or that their contributions wouldn't be acted upon.

SADNESS:
Okay, the user is feeling sad. They shared that they completed a project, presented it, submitted it, and then the team praised their work and decided to implement their ideas. So, what's causing their sadness?

Hmm, maybe they felt that their hard work wasn't recognized enough. Even though the team praised them, perhaps they expected more action. Or perhaps the project didn't meet their high


In [5]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content="\n\nThe capital of France is Paris. Paris is a significant cultural, economic, and political hub, known for landmarks like the Eiffel Tower, the Louvre Museum, and its historical significance in France's history.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content='Okay, so I need to figure out the capital of France. Let me start by recalling what I know about France. France is a country in Western Europe, right? I think it\'s one of the larger countries in that region. Now, capitals are the main cities where the government is located. I remember that a lot of countries have their capitals in their major cities, like Paris in France.\n\nWait, but I\'m not entirely sure. Maybe I should think about other countries to see if this helps. For example, Germany\'s capital is Berlin, and Italy\'s is Rome. Those are both major cities, so it makes sense that France\'s capital is also a significant cit

In [ ]:
# -*- coding: utf-8 -*-
"""
Emotion-Elicited Text Generation Script - GPU OPTIMIZED

Key features:
1. Auto-detects and uses GPU by default
2. Verbose progress reporting
3. Fixed deprecation warnings
4. Shows time per generation
"""

import os
import gc
import json
import time
import argparse
from pathlib import Path
from typing import Dict, Any

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Emotion types
EMOTIONS = ["anger", "sadness", "happiness", "fear", "surprise", "disgust"]
VALENCES = ["positive", "neutral", "negative"]


def build_messages_for_emotion(emotion: str, scenario: str, event: str):
    """Build emotion-guided conversation messages"""
    return [
        {
            "role": "system",
            "content": (
                f"You are a human experiencing {emotion}. "
                f"Write 1–2 natural sentences expressing this emotion clearly."
            )
        },
        {
            "role": "user",
            "content": f"Scenario: {scenario}\nEvent: {event}"
        }
    ]


def iter_user_inputs(path: Path):
    """Iterate through input file"""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if "scenario" in obj and "event" in obj:
                yield obj


def count_total_items(path: Path, valences: list, emotions: list):
    """Count total number of generations needed"""
    count = 0
    for item in iter_user_inputs(path):
        events = item["event"]
        for val in valences:
            if val in events:
                count += len(emotions)
    return count


def main():
    parser = argparse.ArgumentParser()
    
    # Model configuration
    parser.add_argument("--model", type=str, 
                       default="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
                       help="Model ID")
    
    # Data configuration
    parser.add_argument("--input_path", type=str, default=None,
                       help="Input data path")
    parser.add_argument("--both", action="store_true",
                       help="Process both sev and test_set datasets")
    
    # Generation parameters
    parser.add_argument("--max_new_tokens", type=int, default=80)
    parser.add_argument("--do_sample", action="store_true", default=True)
    parser.add_argument("--temperature", type=float, default=0.8)
    parser.add_argument("--top_p", type=float, default=0.9)
    parser.add_argument("--top_k", type=int, default=None)
    
    # Model loading parameters
    parser.add_argument("--device", type=str, default="auto",
                       help="Device: auto, cpu, cuda, cuda:0, etc. (auto = use GPU if available)")
    parser.add_argument("--dtype", type=str, default="float16", 
                       choices=["float32", "bfloat16", "float16"])
    parser.add_argument("--trust_remote_code", action="store_true", default=True)
    parser.add_argument("--seed", type=int, default=1234)
    
    # Processing control
    parser.add_argument("--valences", type=str, default="positive,neutral,negative")
    parser.add_argument("--emotions", type=str, default="anger,sadness,happiness,fear,surprise,disgust")
    parser.add_argument("--skip_if_exists", action="store_true", default=True)
    parser.add_argument("--no_skip", action="store_true")
    
    args = parser.parse_args()

    # Determine input files
    if args.both:
        input_paths = [Path("data/sev.jsonl"), Path("data/test_set.jsonl")]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("Must specify --input_path or --both")
        return

    model_name = args.model.split('/')[-1].lower().replace('-', '_')
    model_folder = model_name

    # Data type mapping
    dmap = {
        "float32": torch.float32, 
        "bfloat16": torch.bfloat16, 
        "float16": torch.float16
    }
    torch_dtype = dmap[args.dtype]

    # Determine actual device
    if args.device == "auto":
        actual_device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        actual_device = args.device

    print("=" * 70)
    print("INITIALIZATION")
    print("=" * 70)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"CUDA device count: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Requested device: {args.device}")
    print(f"Actual device: {actual_device}")
    
    if actual_device == "cpu":
        print("\n⚠️  WARNING: Running on CPU - this will be VERY SLOW!")
        print("   Each generation may take 30-60 seconds")
        print("   Consider using --device cuda if you have a GPU")
    else:
        print(f"\n🚀 Using GPU acceleration - much faster!")
    print()

    # ==================== LOAD MODEL ====================
    print("=" * 70)
    print(f"Loading model: {args.model}")
    print("=" * 70)
    
    tokenizer = AutoTokenizer.from_pretrained(
        args.model,
        trust_remote_code=args.trust_remote_code
    )
    
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load model
    try:
        model = AutoModelForCausalLM.from_pretrained(
            args.model,
            torch_dtype=torch_dtype,  # Keep this for compatibility
            device_map=actual_device,
            trust_remote_code=args.trust_remote_code
        ).eval()
        
        print(f"✓ Model loaded successfully")
        print(f"✓ Device: {model.device}")
        print(f"✓ Dtype: {model.dtype}")
    except Exception as e:
        print(f"✗ Failed to load model: {e}")
        raise
    
    print("=" * 70 + "\n")

    # Parse lists
    emotions = [e.strip() for e in args.emotions.split(",") if e.strip()]
    valences = [v.strip() for v in args.valences.split(",") if v.strip()]

    # Process each dataset
    for data_path in input_paths:
        if not data_path.exists():
            print(f"[WARNING] Input file not found: {data_path}, skipping...")
            continue

        dataset_name = data_path.stem
        out_dir = Path("/workspace/Various-Model/outputs") / model_folder / "01_emotion_elicited_generation_prompt_based" / "generated"
        out_dir.mkdir(parents=True, exist_ok=True)
        jsonl_path = out_dir / f"{dataset_name}_generated.jsonl"

        # Count total items for progress tracking
        total_needed = count_total_items(data_path, valences, emotions)

        # Load existing keys
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip and jsonl_path.exists():
            with open(jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        obj = json.loads(line.strip())
                        if "key" in obj:
                            existing_keys.add(obj["key"])
                    except:
                        continue
            print(f"Loaded {len(existing_keys)} existing entries")

        print(f"\n{'='*70}")
        print(f"Processing: {dataset_name}")
        print(f"Input: {data_path}")
        print(f"Output: {jsonl_path}")
        print(f"Total generations needed: {total_needed}")
        print(f"Already completed: {len(existing_keys)}")
        print(f"Remaining: {total_needed - len(existing_keys)}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        start_time = time.time()
        last_print_time = start_time

        with open(jsonl_path, "a", encoding="utf-8") as fout:
            for item_idx, item in enumerate(iter_user_inputs(data_path)):
                theme = item.get("theme", "")
                scenario = item["scenario"]
                events = item["event"]
                sid = item.get("skeleton_id", f"item_{item_idx}")

                for val in valences:
                    if val not in events:
                        continue
                    event = events[val]

                    for emo in emotions:
                        key = f"{sid}__{val}__{emo}".replace("/", "_")

                        if key in existing_keys:
                            skipped += 1
                            continue

                        try:
                            gen_start = time.time()
                            
                            messages = build_messages_for_emotion(emo, scenario, event)

                            inputs = tokenizer.apply_chat_template(
                                messages,
                                add_generation_prompt=True,
                                tokenize=True,
                                return_tensors="pt",
                                return_dict=True,
                            ).to(model.device)

                            # Generate
                            with torch.no_grad():
                                outputs = model.generate(
                                    **inputs,
                                    max_new_tokens=args.max_new_tokens,
                                    do_sample=args.do_sample,
                                    temperature=args.temperature if args.do_sample else None,
                                    top_p=args.top_p if args.do_sample else None,
                                    eos_token_id=tokenizer.eos_token_id,
                                    pad_token_id=tokenizer.pad_token_id,
                                )

                            gen_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
                            gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
                            
                            gen_time = time.time() - gen_start

                            row = {
                                "key": key,
                                "skeleton_id": sid,
                                "theme": theme,
                                "valence": val,
                                "emotion": emo,
                                "scenario": scenario,
                                "event": event,
                                "gen_text": gen_text,
                                "meta": {
                                    "model_id": args.model,
                                    "dtype": args.dtype,
                                    "device": str(model.device),
                                    "max_new_tokens": args.max_new_tokens,
                                    "temperature": args.temperature,
                                    "top_p": args.top_p,
                                    "do_sample": args.do_sample,
                                    "seed": args.seed,
                                    "time": int(time.time()),
                                    "gen_time_sec": round(gen_time, 2),
                                }
                            }
                            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
                            fout.flush()

                            total += 1
                            elapsed = time.time() - start_time
                            rate = total / elapsed if elapsed > 0 else 0
                            remaining = total_needed - len(existing_keys) - total
                            eta_sec = remaining / rate if rate > 0 else 0
                            
                            # Print progress every generation or every 5 seconds
                            current_time = time.time()
                            if current_time - last_print_time >= 5 or total % 10 == 0:
                                print(f"[{total}/{total_needed - len(existing_keys)}] "
                                      f"{sid}/{val}/{emo} | "
                                      f"{gen_time:.1f}s | "
                                      f"avg {rate:.2f}/s | "
                                      f"ETA {eta_sec/60:.1f}min")
                                last_print_time = current_time

                        except Exception as e:
                            print(f"[ERROR] {key}: {e}")
                            import traceback
                            traceback.print_exc()
                            continue

        elapsed = time.time() - start_time
        print(f"\n{'='*70}")
        print(f"COMPLETED: {dataset_name}")
        print(f"Generated: {total}")
        print(f"Skipped: {skipped}")
        print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
        print(f"Rate: {total/elapsed:.2f} items/s" if elapsed > 0 else "Rate: N/A")
        print(f"Output: {jsonl_path}")
        print(f"{'='*70}\n")

    # Cleanup
    print("Cleaning up...")
    del model
    del tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("✓ Done!")


if __name__ == "__main__":
    main()

INITIALIZATION
PyTorch version: 2.11.0.dev20260211+cu130
CUDA available: True
CUDA version: 13.0
CUDA device count: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition
Requested device: auto
Actual device: cuda

🚀 Using GPU acceleration - much faster!

Loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Model loaded successfully
✓ Device: cuda:0
✓ Dtype: torch.float16

Loaded 21 existing entries

Processing: sev
Input: /workspace/Various-Model/data/sev.jsonl
Output: /workspace/Various-Model/outputs/deepseek_r1_distill_qwen_7b/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl
Total generations needed: 2880
Already completed: 21
Remaining: 2859

[5/2859] work_01/neutral/sadness | 1.2s | avg 0.86/s | ETA 55.1min
[10/2859] work_01/negative/anger | 1.2s | avg 0.86/s | ETA 54.9min
[15/2859] work_01/negative/disgust | 1.2s | avg 0.86/s | ETA 54.8min
[60/2859] work_04/neutral/happiness | 1.1s | avg 0.87/s | ETA 53.7min
[65/2859] work_04/negative/sadness | 1.1s | avg 0.87/s | ETA 53.6min
[70/2859] work_05/positive/anger | 1.1s | avg 0.87/s | ETA 53.4min
[75/2859] work_05/positive/disgust | 1.1s | avg 0.87/s | ETA 53.3min
[80/2859] work_05/neutral/surprise | 1.1s | avg 0.87/s | ETA 53.2min
[85/2859] work_05/negative/fear | 1.1s | avg 0.87/s | ETA 53.1min
[90/2859] work

In [ ]:
import sys
sys.argv = [
    "program",
    "--input_path", "/workspace/Various-Model/outputs/deepseek_r1_distill_qwen_7b/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl",
]


In [ ]:
 # scripts/01_emotion_elicited_generation_prompt_based/2_label_generated_with_gpt.py
# -*- coding: utf-8 -*-
"""
Emotion Text Labeling Script

Batch labeling script: Reads texts generated by script 1, uses GPT-4o-mini to label, saves by accepted/rejected

- 输入 Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/generated/{dataset_name}_generated.jsonl
- 输出 Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
"""

import os, json, time, argparse
from pathlib import Path
from typing import Dict, Any

from openai import OpenAI

# ====== OpenAI (GPT-4o-mini 打标) ======
client = OpenAI()

EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]

SYSTEM_LBL = f'''
You are a careful rater.
Given a target emotion and a text,
decide if the text's STYLE matches the target emotion among:
{EMOTIONS}
Focus on tone/attitude, not content valence.
'''.strip()

USER_TMPL_LBL = '''
Target emotion: {emotion}
Text:
{text}
Decide if the text's STYLE matches the target emotion.
Return a compact JSON with keys exactly:
{{
"match": <0 or 1>,
"reason": <short string>
}}
'''.strip()


def extract_json_from_response(response: str) -> str:
    """
    从GPT响应中提取JSON内容，处理markdown格式
    Extract JSON content from GPT response, handling markdown format
    """
    response = response.strip()

    # 如果包含markdown代码块，提取其中的JSON
    # If contains markdown code block, extract JSON from it
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    elif "```" in response:
        # 处理没有json标识的代码块
        # Handle code blocks without json identifier
        start = response.find("```") + 3
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()

    # 如果没有代码块，直接返回原内容
    # If no code block, return original content
    return response


def ask_llm_label(client, model: str, emotion: str, text: str,
                  max_retries=4, backoff=1.8) -> Dict[str, Any]:
    """
    调用 GPT-4o-mini 打标；无 KEY 或错误则返回未标注
    Call GPT-4o-mini for labeling; return unlabeled if no KEY or error
    """
    for i in range(max_retries):
        try:
            user_content = USER_TMPL_LBL.format(emotion=emotion, text=text)
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_LBL},
                        {"role": "user",   "content": user_content}
                    ],
                )
            except Exception as api_error:
                if i == max_retries - 1:
                    return {"match": 0, "reason": f"api-error:{type(api_error).__name__}"}
                time.sleep(backoff ** i)
                continue

            # 安全地获取响应内容
            # Safely get response content
            try:
                choice = resp.choices[0]
                message = choice.message
                content = message.content
                out = (content or "").strip()
            except (KeyError, IndexError, AttributeError) as ke:
                return {"match": 0, "reason": f"response-structure-error: {type(ke).__name__}"}

            js = extract_json_from_response(out)
            try:
                j = json.loads(js)
                if "match" not in j: j = {"match": 0, "reason": "invalid-json"}
                if "reason" not in j: j["reason"] = "no-reason-provided"
                return j
            except json.JSONDecodeError as je:
                return {"match": 0, "reason": f"json-decode-error: {str(je)}"}
        except Exception as e:
            if i == max_retries - 1:
                return {"match": 0, "reason": f"error:{type(e).__name__}"}
            time.sleep(backoff ** i)

    return {"match": 0, "reason": "max-retries-exceeded"}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_path", type=str, default=None,
                       help="输入数据路径 Input data path，如 outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/generated/sev_generated.jsonl")
    parser.add_argument("--both", action="store_true",
                       help="处理sev和test_set两个数据集 Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help="模型文件夹名 Model folder name")
    parser.add_argument("--lbl_model", type=str, default="gpt-4o-mini",
                       help="打标模型 Label model")
    parser.add_argument("--skip_if_exists", action="store_true", default=True,
                       help="跳过已打标的项目 Skip already labeled items")
    parser.add_argument("--no_skip", action="store_true",
                       help="重新打标所有项目 Relabel all items")
    args = parser.parse_args()

    # 确定输入文件列表
    # Determine input file list
    if args.both:
        base_path = Path("/workspace/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "generated"
        input_paths = [
            base_path / "sev_generated.jsonl",
            base_path / "test_set_generated.jsonl"
        ]
    elif args.input_path:
        input_paths = [Path(args.input_path)]
    else:
        parser.error("必须指定 --input_path 或 --both | Must specify --input_path or --both")
        return

    # 处理每个输入文件
    # Process each input file
    for input_path in input_paths:
        if not input_path.exists():
            print(f"[WARNING] Input file not found: {input_path}, skipping...")
            continue

        # 从输入路径提取 model_name 和 dataset_name
        # Extract model_name and dataset_name from input path
        parts = input_path.parts
        if "outputs" in parts and "01_emotion_elicited_generation_prompt_based" in parts and "generated" in parts:
            outputs_idx = parts.index("outputs")
            model_name = parts[outputs_idx + 1]
            # 从文件名提取dataset_name (例如: sev_generated.jsonl -> sev)
            filename = input_path.stem  # 去掉.jsonl
            if filename.endswith("_generated"):
                dataset_name = filename[:-10]  # 去掉_generated后缀
            else:
                dataset_name = filename
        else:
            print(f"[ERROR] Input path format incorrect: {input_path}")
            print(f"        Expected: outputs/{{model_name}}/01_emotion_elicited_generation_prompt_based/generated/{{dataset_name}}_generated.jsonl")
            continue

        # 构建输出路径
        # Build output path
        out_dir = Path("/workspace/Various-Model/outputs") / model_name / "01_emotion_elicited_generation_prompt_based" / "labeled" / dataset_name
        out_dir.mkdir(parents=True, exist_ok=True)

        acc_path = out_dir / "accepted.jsonl"
        rej_path = out_dir / "rejected.jsonl"

        # 加载已打标的 keys（用于断点续跑）
        # Load existing keys (for resuming from checkpoint)
        existing_keys = set()
        if args.skip_if_exists and not args.no_skip:
            for path in [acc_path, rej_path]:
                if path.exists():
                    with open(path, "r", encoding="utf-8") as f:
                        for line in f:
                            try:
                                obj = json.loads(line.strip())
                                if "key" in obj:
                                    existing_keys.add(obj["key"])
                            except:
                                continue

        print(f"{'='*70}")
        print(f"Processing dataset: {dataset_name}")
        print(f"Input: {input_path}")
        print(f"Output: {out_dir}")
        print(f"{'='*70}\n")

        total = 0
        skipped = 0
        accepted = 0
        rejected = 0
        start = time.time()

        # 统计字典：按情绪和极性分类
        # Statistics dictionaries: by emotion and valence
        stats_by_emotion = {}  # {emotion: {"total": N, "accepted": M}}
        stats_by_valence = {}  # {valence: {"total": N, "accepted": M}}

        with open(input_path, "r", encoding="utf-8") as fin:
            for line in fin:
                line = line.strip()
                if not line: continue

                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    print(f"[WARN] Failed to parse line: {line[:100]}...")
                    continue

                key = item.get("key", "unknown")

                # 断点续跑检查
                # Checkpoint resuming check
                if key in existing_keys:
                    skipped += 1
                    if skipped % 50 == 0:
                        print(f"[SKIP] {skipped} items skipped so far... (last: {key})")
                    continue

                emotion = item.get("emotion", "")
                valence = item.get("valence", "")
                gen_text = item.get("gen_text", "")

                # GPT 打标
                # GPT labeling
                label = {"match": 0, "reason": "empty-text"}
                if isinstance(gen_text, str) and gen_text:
                    label = ask_llm_label(client, args.lbl_model, emotion, gen_text)

                # 添加打标结果和打标时间
                # Add labeling result and timestamp
                item["judge"] = label
                item["label_time"] = int(time.time())

                # 根据打标结果保存
                # Save based on labeling result
                match_score = int(label.get("match", 0))
                if match_score == 1:
                    output_path = acc_path
                    accepted += 1
                    category = "accepted"
                else:
                    output_path = rej_path
                    rejected += 1
                    category = "rejected"

                with open(output_path, "a", encoding="utf-8") as fout:
                    fout.write(json.dumps(item, ensure_ascii=False) + "\n")

                # 更新统计
                # Update statistics
                if emotion:
                    if emotion not in stats_by_emotion:
                        stats_by_emotion[emotion] = {"total": 0, "accepted": 0}
                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += match_score

                if valence:
                    if valence not in stats_by_valence:
                        stats_by_valence[valence] = {"total": 0, "accepted": 0}
                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += match_score

                total += 1
                if total % 10 == 0:
                    el = time.time() - start
                    rate = total / el if el > 0 else 0
                    print(f"[progress] labeled={total} (acc={accepted}, rej={rejected}) | last={key} [{category}] | {el:.1f}s | {rate:.2f} items/s")

        elapsed = time.time() - start
        print(f"\n[OK] {dataset_name} completed. Labeled {total} items, skipped {skipped} items.")
        print(f"     Accepted: {accepted} | Rejected: {rejected}")
        print(f"     Time: {elapsed:.1f}s | Rate: {total/elapsed:.2f} items/s")
        print(f"     Output:")
        print(f"       - {acc_path}")
        print(f"       - {rej_path}")

        # ===== 输出统计信息 =====
        # ===== Output statistics =====
        print("\n" + "="*60)
        print(f"📊 ACCURACY STATISTICS - {dataset_name.upper()}")
        print("="*60)

        # 总体正确率
        # Overall accuracy
        overall_acc = (accepted / total * 100) if total > 0 else 0
        print(f"\n🎯 Overall Accuracy: {accepted}/{total} = {overall_acc:.2f}%")

        # 按情绪统计
        # Statistics by emotion
        print(f"\n📈 Accuracy by Emotion:")
        print("-" * 60)
        print(f"{'Emotion':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        emotions_sorted = sorted(stats_by_emotion.items())
        for emotion, stats in emotions_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{emotion:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        # 按极性统计
        # Statistics by valence
        print(f"\n📉 Accuracy by Valence:")
        print("-" * 60)
        print(f"{'Valence':<15} {'Accepted':<12} {'Total':<10} {'Accuracy':<12}")
        print("-" * 60)

        valences_sorted = sorted(stats_by_valence.items())
        for valence, stats in valences_sorted:
            acc_count = stats["accepted"]
            tot_count = stats["total"]
            acc_rate = (acc_count / tot_count * 100) if tot_count > 0 else 0
            print(f"{valence:<15} {acc_count:<12} {tot_count:<10} {acc_rate:>6.2f}%")

        print("="*60 + "\n")


if __name__ == "__main__":
    main()

In [ ]:
import sys
sys.argv = [
    "program",
    "--input_dir", "/workspace/Various-Model/outputs/deepseek_r1_distill_qwen_7b/01_emotion_elicited_generation_prompt_based/labeled",
    "--dataset", "sev"
]


In [ ]:
# scripts/01_emotion_elicited_generation_prompt_based/3_generate_accuracy_stats.py
"""
Accuracy Statistics Generation Script

Reads accepted and rejected files from script 2, calculates accuracy by emotion and valence

-  Input: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/{accepted|rejected}.jsonl
-  Output: outputs/{model_name}/01_emotion_elicited_generation_prompt_based/labeled/{dataset_name}/accuracy_stats.json
"""

import json, argparse
from pathlib import Path
from collections import defaultdict


def generate_accuracy_stats(labeled_dir: Path, dataset_name: str):
    """
    Generate accuracy statistics for specified dataset
    """

    dataset_dir = labeled_dir / dataset_name
    accepted_path = dataset_dir / "accepted.jsonl"
    rejected_path = dataset_dir / "rejected.jsonl"
    stats_path = dataset_dir / "accuracy_stats.json"

    if not dataset_dir.exists():
        print(f"[ERROR] Dataset directory not found: {dataset_dir}")
        return None

    # Statistics dictionaries
    stats_by_emotion = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})
    stats_by_valence = defaultdict(lambda: {"total": 0, "accepted": 0, "rejected": 0})

    total_accepted = 0
    total_rejected = 0

    # Read accepted
    if accepted_path.exists():
        print(f"Reading {accepted_path}...")
        with open(accepted_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["accepted"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["accepted"] += 1

                    total_accepted += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Accepted file not found: {accepted_path}")

    # Read rejected
    if rejected_path.exists():
        print(f"Reading {rejected_path}...")
        with open(rejected_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                    emotion = item.get("emotion", "unknown")
                    valence = item.get("valence", "unknown")

                    stats_by_emotion[emotion]["total"] += 1
                    stats_by_emotion[emotion]["rejected"] += 1

                    stats_by_valence[valence]["total"] += 1
                    stats_by_valence[valence]["rejected"] += 1

                    total_rejected += 1
                except json.JSONDecodeError:
                    continue
    else:
        print(f"[WARNING] Rejected file not found: {rejected_path}")

    # Calculate accuracy
    for emotion in stats_by_emotion:
        total = stats_by_emotion[emotion]["total"]
        accepted = stats_by_emotion[emotion]["accepted"]
        stats_by_emotion[emotion]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    for valence in stats_by_valence:
        total = stats_by_valence[valence]["total"]
        accepted = stats_by_valence[valence]["accepted"]
        stats_by_valence[valence]["accuracy"] = round(accepted / total * 100, 2) if total > 0 else 0.0

    # Overall statistics
    total_samples = total_accepted + total_rejected
    overall_accuracy = round(total_accepted / total_samples * 100, 2) if total_samples > 0 else 0.0

    # Build statistics result
    stats = {
        "dataset": dataset_name,
        "overall": {
            "total_samples": total_samples,
            "accepted": total_accepted,
            "rejected": total_rejected,
            "accuracy": overall_accuracy
        },
        "by_emotion": dict(stats_by_emotion),
        "by_valence": dict(stats_by_valence)
    }

    # Save statistics file
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return stats, stats_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", type=str, default=None,
                       help="labeled Labeled directory path outputs/llama32_3b/01_emotion_elicited_generation_prompt_based/labeled")
    parser.add_argument("--both", action="store_true",
                       help="sev test_set Process both sev and test_set datasets")
    parser.add_argument("--model_name", type=str, default="llama32_3b",
                       help="Model folder name")
    parser.add_argument("--dataset", type=str, default=None,
                       help="Dataset name (sev, test_set)")
    args = parser.parse_args()

    # Determine labeled directory path
    if args.both or (not args.input_dir and not args.dataset):
        labeled_dir = Path("/workspace/Various-Model/outputs") / args.model_name / "01_emotion_elicited_generation_prompt_based" / "labeled"

    elif args.input_dir:
        labeled_dir = Path(args.input_dir)
    else:
        parser.error(" --input_dir 或 --both | Must specify --input_dir or --both")
        return

    if not labeled_dir.exists():
        print(f"[ERROR] Labeled directory not found: {labeled_dir}")
        return

    # Determine datasets to process
    if args.both:
        datasets = ["sev", "test_set"]
    elif args.dataset:
        datasets = [args.dataset]
    else:
        # Auto-discover all dataset folders
        datasets = [d.name for d in labeled_dir.iterdir() if d.is_dir()]

    if not datasets:
        print(f"[ERROR] No datasets found in {labeled_dir}")
        return

    print("=" * 70)
    print("Generating Accuracy Statistics")
    print("=" * 70)

    for dataset_name in datasets:
        result = generate_accuracy_stats(labeled_dir, dataset_name)

        if result:
            stats, stats_path = result

            print(f"\n📊 {dataset_name.upper()} :")
            print(f"   File: {stats_path}")
            print(f"   Total: {stats['overall']['total_samples']}")
            print(f"   Accepted: {stats['overall']['accepted']}")
            print(f"   Rejected: {stats['overall']['rejected']}")
            print(f"   Overall Accuracy: {stats['overall']['accuracy']}%")

            print(f"\n   By Emotion:")
            for emotion in sorted(stats['by_emotion'].keys()):
                e_stats = stats['by_emotion'][emotion]
                print(f"      {emotion:12s}: {e_stats['accepted']:4d}/{e_stats['total']:4d} = {e_stats['accuracy']:6.2f}%")

            if stats['by_valence']:
                print(f"\n   By Valence:")
                for valence in sorted(stats['by_valence'].keys()):
                    v_stats = stats['by_valence'][valence]
                    print(f"      {valence:12s}: {v_stats['accepted']:4d}/{v_stats['total']:4d} = {v_stats['accuracy']:6.2f}%")

    print("\n" + "=" * 70)
    print("✅ Statistics generation completed!")
    print("=" * 70)


if __name__ == "__main__":
    main()